# Mission 4: Mini-Google & Name Generator 🔎

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anshid/MathMeetsML/blob/main/notebooks/04_mini_google.ipynb)

> **From your syllabus**
> - *Applied Probability*, Unit IV (Markov chains: Chapman-Kolmogorov, limiting probabilities) and Unit II (law of large numbers)
> - *Linear Algebra*: eigenvectors

**Your mission:** rank web pages the way Google once did, then make a machine invent new names. Both are the same object: a **Markov chain**.

> 📊 **Progress tracking: please read.** The next cell draws your progress map and records which exercises you finish. It sends your **name, roll number, exercise ID and time** to the instructor's progress sheet for this workshop, nothing else (no files, nothing from your Google account). Its full code is in the cell, so you can read it. Prefer not to be tracked? Type `anonymous` as your name, or set `TRACKING = False`; the map still works for you.
>
> Colab may ask *"Do you trust this code?"* because this notebook comes from GitHub (`anshid/MathMeetsML`, open source, MIT licence). That is expected: choose **Run anyway**.

In [ ]:
#@title Progress map (run me first; the code is short, read it)
# Records which exercises you finish so you can see your progress map and the instructor can see how the class is doing.
# Sent: your name, roll number, exercise id and time. Nothing else. Type "anonymous" as your name (or set TRACKING = False) to send nothing.
TRACKING = True
MISSION = 4

import json, os, time, base64, urllib.parse, urllib.request
from IPython.display import HTML, display, update_display

CLOUD_URL = "https://script.google.com/macros/s/AKfycbxekbOkIE5oRVA319mhEENS1phVLt7gPfRUernniP8kMopgIqUNHNN37xjF_3XZEuQWwQ/exec"
TOKEN = "3F5y-xKcPbZZPT49"
LAN_URL = os.environ.get("MML_LAN", "")
MISSIONS = {"0": {"title": "Ignite", "checkpoints": ["m00-ready", "m00-done"]}, "1": {"title": "Golf", "checkpoints": ["m01-grad", "m01-done"]}, "2": {"title": "Digit Reader", "checkpoints": ["m02-grad", "m02-done"]}, "3": {"title": "Selfie", "checkpoints": ["m03-ey", "m03-pca", "m03-done"]}, "4": {"title": "Mini-Google", "checkpoints": ["m04-markov", "m04-done"]}, "5": {"title": "Tiny AI", "checkpoints": ["m05-backprop", "m05-done"]}}
STORE = "progress.json"
SINKS = [u for u in (CLOUD_URL, LAN_URL) if u]


def _load():
    try:
        with open(STORE) as f:
            return json.load(f)
    except Exception:
        return {"name": "", "roll": "", "done": {}, "queue": []}


def _save(s):
    try:
        with open(STORE, "w") as f:
            json.dump(s, f)
    except Exception:
        pass


def _who(s):
    if s["roll"]:
        return
    if os.environ.get("MML_TRACKING") == "off":
        s["name"], s["roll"] = "anonymous", "-"
        return
    try:
        s["name"] = input("Your name (or 'anonymous'): ").strip() or "anonymous"
        s["roll"] = input("Roll number: ").strip() or "-"
    except Exception:
        s["name"], s["roll"] = "anonymous", "-"


def _sending(s):
    return TRACKING and s["name"].lower() != "anonymous" and os.environ.get("MML_TRACKING") != "off"


def _post(url, ev):
    body = json.dumps(dict(ev, token=TOKEN)).encode()
    urllib.request.urlopen(urllib.request.Request(url, body, {"Content-Type": "text/plain"}), timeout=2).read()


def sync():
    """Send queued exercises to the instructor. Silent if offline; they stay queued."""
    s = _load()
    if _sending(s):
        dead = set()
        for ev in s["queue"]:
            for u in SINKS:
                if u in dead or u in ev["sent"]:
                    continue
                try:
                    _post(u, ev)
                    ev["sent"].append(u)
                except Exception:
                    dead.add(u)
        s["queue"] = [e for e in s["queue"] if any(u not in e["sent"] for u in SINKS)]
    _save(s)
    return s


def progress_code():
    """Offline? Paste this code to the instructor (chat or USB) instead."""
    s = _load()
    print(base64.b64encode(json.dumps({"name": s["name"], "roll": s["roll"], "done": s["done"]}).encode()).decode())


def _status(s, n):
    ids = MISSIONS[str(n)]["checkpoints"]
    got = sum(1 for c in ids if c in s["done"])
    if got == len(ids):
        return "done", got, len(ids)
    if n == MISSION or got:
        return "now", got, len(ids)
    return "todo", got, len(ids)


def _html(s):
    cols = {"done": "#2e9e5b", "now": "#3b82f6", "todo": "#8a8f98"}
    parts, stats = [], [_status(s, n) for n in range(len(MISSIONS))]
    for n, (st, got, tot) in enumerate(stats):
        x = 45 + n * 108
        title = MISSIONS[str(n)]["title"]
        if n:
            ok = stats[n - 1][0] == "done"
            parts.append(f'<line x1="{x - 108 + 20}" y1="40" x2="{x - 20}" y2="40" stroke="{cols["done"] if ok else cols["todo"]}" stroke-width="3" stroke-dasharray="{"0" if ok else "5 4"}"/>')
        if st == "done":
            parts.append(f'<circle cx="{x}" cy="40" r="18" fill="{cols[st]}"/><path d="M{x - 8} 40 l6 7 l11 -14" stroke="#fff" stroke-width="3.5" fill="none" stroke-linecap="round" stroke-linejoin="round"/>')
        elif st == "now":
            parts.append(f'<circle cx="{x}" cy="40" r="18" fill="none" stroke="{cols[st]}" stroke-width="4"><animate attributeName="r" values="18;22;18" dur="1.8s" repeatCount="indefinite"/></circle><text x="{x}" y="45" text-anchor="middle" font-size="14" font-weight="700" fill="{cols[st]}">{got}/{tot}</text>')
        else:
            parts.append(f'<circle cx="{x}" cy="40" r="18" fill="none" stroke="{cols[st]}" stroke-width="2.5" stroke-dasharray="4 3"/><text x="{x}" y="45" text-anchor="middle" font-size="14" fill="{cols[st]}">{n}</text>')
        parts.append(f'<text x="{x}" y="82" text-anchor="middle" font-size="12" fill="currentColor" opacity="{1 if st != "todo" else 0.55}">M{n} {title}</text>')
    finished = [n for n, t in enumerate(stats) if t[0] == "done"]
    todo = [f"M{n}" for n, t in enumerate(stats) if t[0] == "todo"]
    line = f"<b>{len(finished)} of {len(stats)} missions finished</b> &middot; now in <b>Mission {MISSION}</b> ({stats[MISSION][1]}/{stats[MISSION][2]} exercises)"
    line += f" &middot; not visited yet: {', '.join(todo)}" if todo else " &middot; every mission visited"
    who = f"{s['name']} ({s['roll']})" if s["roll"] else ""
    return (f'<div style="font-family:sans-serif;max-width:700px"><svg viewBox="0 0 640 96" width="100%" role="img" aria-label="Progress map">{"".join(parts)}</svg>'
            f'<div style="font-size:14px">{line}</div><div style="font-size:12px;opacity:.6">{who}</div></div>')


_shown = []


def show_tracker():
    html = HTML(_html(_load()))
    if _shown:
        try:
            update_display(html, display_id="mml-tracker")
            return
        except Exception:
            pass
    display(html, display_id="mml-tracker")
    _shown.append(1)


def checkpoint(cid):
    """Call after an exercise passes: saves it, tells the instructor (if online), refreshes the map."""
    s = _load()
    _who(s)
    if cid not in s["done"]:
        s["done"][cid] = time.strftime("%H:%M:%S")
        s["queue"].append({"name": s["name"], "roll": s["roll"], "mission": MISSION, "checkpoint": cid, "ts": s["done"][cid], "sent": []})
    _save(s)
    sync()
    show_tracker()


def _start():
    s = _load()
    _who(s)
    if CLOUD_URL and _sending(s):  # Colab gives every notebook a fresh disk, so ask what you already finished elsewhere
        try:
            q = urllib.parse.urlencode({"roll": s["roll"], "token": TOKEN})
            for c in json.load(urllib.request.urlopen(CLOUD_URL + "?" + q, timeout=2)).get("done", []):
                s["done"].setdefault(c, "earlier")
        except Exception:
            pass
    _save(s)
    show_tracker()


_start()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
pages = ["Home", "Maths", "ML", "Python", "Blog", "Jobs"]
# A[i, j] = 1 means page i links to page j
A = np.array([
    [0, 1, 1, 1, 0, 0],
    [1, 0, 1, 0, 0, 0],
    [1, 1, 0, 1, 0, 1],
    [0, 0, 1, 0, 1, 0],
    [1, 0, 0, 0, 0, 1],
    [1, 0, 1, 0, 0, 0],
], dtype=float)
print("links out of each page:", A.sum(axis=1))

## 1. The random surfer
A surfer sits on a page and clicks a random outgoing link. That is a Markov chain with transition matrix
$P_{ij}=A_{ij}/\sum_kA_{ik}$ (each row sums to 1). After $n$ clicks the distribution is $\pi_n=\pi_0P^n$ (**Chapman-Kolmogorov**).

**Core 1:** build $P$ from $A$.

<details><summary>Hint</summary>

Divide each row by its sum: `A / A.sum(axis=1, keepdims=True)`.
</details>

In [ ]:
P = ____                           # TODO: row-normalise A
print(np.round(P, 2))
print("rows sum to 1:", np.allclose(P.sum(axis=1), 1))

## 2. Where does the surfer spend time?
The **limiting distribution** $\pi$ satisfies $\pi=\pi P$ (a left eigenvector with eigenvalue 1). Page importance = long-run fraction of time spent there. To guarantee a unique limit, Google's PageRank adds a **teleport**: with probability $1-\alpha$ jump to a uniformly random page,
$$G=\alpha P+\frac{1-\alpha}{n}\mathbf 1\mathbf 1^\top,\qquad \alpha=0.85 .$$
$G$ has all entries positive, so the chain is irreducible and aperiodic and the limit exists (Applied Probability Unit IV).

**Core 2:** power iteration $\pi\leftarrow\pi G$ starting from the uniform distribution.

In [ ]:
n = len(pages)
alpha = 0.85
G = alpha * P + (1 - alpha) / n * np.ones((n, n))

pi = np.ones(n) / n
for step in range(100):
    pi = ____                      # TODO: one step of the chain, pi times G
print({p: round(float(v), 3) for p, v in zip(pages, pi)})

**Check three ways.** (1) $\pi G=\pi$; (2) $\pi$ is the eigenvector of $G^\top$ for eigenvalue 1; (3) every row of $G^{n}$ converges to $\pi$ (Chapman-Kolmogorov).

In [ ]:
print("1. pi G = pi            :", np.allclose(pi @ G, pi))

vals, vecs = np.linalg.eig(G.T)
v = np.real(vecs[:, np.argmax(np.real(vals))]); v = v / v.sum()
print("2. eigenvector matches  :", np.allclose(v, pi, atol=1e-6), " (eigenvalue", round(np.real(vals).max(), 6), ")")

G50 = np.linalg.matrix_power(G, 50)
print("3. rows of G^50 equal pi:", np.allclose(G50, np.tile(pi, (n, 1)), atol=1e-6))

order = np.argsort(-pi)
plt.bar([pages[i] for i in order], pi[order]); plt.ylabel("PageRank"); plt.show()

## 3. The law of large numbers, live
Simulate one surfer for many clicks. The **fraction of time** on each page should approach $\pi$ (the ergodic theorem, a cousin of the LLN from Applied Probability Unit II).

In [ ]:
steps = 20000
state = 0
visits = np.zeros(n)
for _ in range(steps):
    visits[state] += 1
    state = np.random.choice(n, p=G[state])

print("simulated:", np.round(visits / steps, 3))
print("exact pi :", np.round(pi, 3))
print("max error:", round(np.abs(visits / steps - pi).max(), 4))

**Challenge:** make *Jobs* link to *Blog* only (edit row 5 of `A`) and re-run. Which page loses rank, which gains? Explain using the chain.

## 4. Same maths, new game: a name generator
Treat letters as states. `^` marks the start of a name and `$` the end. Count how often letter $a$ is followed by letter $b$, normalise each row, and you have a Markov chain over letters. Sampling from it **invents** names.

In [ ]:
names = """
aadhya aarav abhinav adithya ajay akhil akshay amal amrita anand anjali anoop anusha arjun arun aswin athira
bhavya devika dhanya dileep divya gayathri gopika hari haritha indu irfan jishnu kavya kiran krishna lakshmi
mahesh meera midhun nandana neha nikhil nimisha pooja prakash priya rahul rohit sachin sanjay sarika shiva
sneha sreya suresh swathi tara varun vignesh vishnu yamuna zahra
""".split()
print(len(names), "training names, e.g.", names[:5])

chars = ["^"] + sorted(set("".join(names))) + ["$"]
idx = {c: i for i, c in enumerate(chars)}
counts = np.zeros((len(chars), len(chars))) + 0.01      # tiny smoothing so no row is all zeros
for name in names:
    seq = ["^"] + list(name) + ["$"]
    for a, b in zip(seq[:-1], seq[1:]):
        counts[idx[a], idx[b]] += 1
print("transition counts:", counts.shape)

**Core 3:** turn counts into transition probabilities (same row-normalising idea as before).

In [ ]:
T = ____                           # TODO: row-normalise counts
assert np.allclose(T.sum(axis=1), 1)
checkpoint("m04-markov")

def invent(rng):
    state, out = idx["^"], ""
    while True:
        state = rng.choice(len(chars), p=T[state])
        if chars[state] == "$" or len(out) > 14:
            return out
        out += chars[state]

rng = np.random.RandomState(7)
print([invent(rng) for _ in range(12)])

In [ ]:
plt.figure(figsize=(7, 6))
plt.imshow(T, cmap="Blues"); plt.xticks(range(len(chars)), chars); plt.yticks(range(len(chars)), chars)
plt.xlabel("next letter"); plt.ylabel("current letter"); plt.title("letter transition matrix"); plt.show()

In [ ]:
checkpoint("m04-done")                      # Mission 4 complete: watch your map update at the top

## Stretch
Use **two** previous letters as the state (a second-order chain) and compare the names. This is exactly the step from a bigram model toward a language model: more context, better text, but many more states.

## Quest (take-home)
Replace the list with 50 names from your own language or family tradition (with permission), generate 20 invented names and pick your favourite. Then compute the stationary distribution of letters and report the three most common letters.

## You just used...
- **Markov chains**, the transition matrix and **Chapman-Kolmogorov** $P^{(n)}=P^n$ (Applied Probability Unit IV)
- **limiting probabilities** as an eigenvector, and why teleportation makes them exist
- the **law of large numbers** for long-run averages (Unit II)
- a **language model in miniature**: predict the next token from the previous one

## Your CV line
> *Implemented PageRank via power iteration and eigen-decomposition and built a character-level Markov chain text generator in NumPy; verified convergence to the stationary distribution by simulation.*

Next: **Mission 5**, look inside a tiny AI.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anshid/MathMeetsML/blob/main/notebooks/05_inside_a_tiny_ai.ipynb)